# 06 · Constrained Decoding at Every Precision

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Goal:** run all 25 model × precision combinations with JSON-schema-constrained decoding (Update 9), to see how much of the quantization penalty it removes (Update 10).

Everything else is the same as the free runs of Updates 4-6: the 1,390 BFCL `non_live` questions, the prompt, greedy decoding, the scorer and the model revisions. The only change is that vLLM's guided decoding (xgrammar) forces every answer to match the schema in `src/schema.py`.

| Precision | Where the weights come from |
|---|---|
| FP16, NF4 | the pinned Hugging Face model (NF4 is quantized by vLLM while loading) |
| INT8 | made here by `src/quantize.py` |
| GPTQ | made here by `src/quantize.py`, then uploaded (private), because the Update 6 checkpoints were not kept |
| AWQ | downloaded from the private Hugging Face repositories made in Update 6 |

**Quick check first:** the first model's AWQ checkpoint is downloaded and answers 2 questions per category in constrained mode.

## How to run on Kaggle

The 25 runs are split into two parts, so that one Kaggle session (12 hours at most) cannot lose all of them. They can run at the same time, in two notebooks.

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file. Do this twice.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. **Add-ons → Secrets:** switch on `HF_TOKEN` and `HF_WRITE_TOKEN` for each notebook.
4. In the settings cell of the second notebook, set `PART = "B"`.
5. Click **Save Version → Save & Run All (Commit)** in both.

**Time:** unknown until it runs. The same work took about 3.3 hours (part A) and 4.8 hours (part B) in free mode, not counting installation (about 20 minutes per part) and the AWQ checkpoints, which are downloaded here. Guided decoding adds time, but how much is not known yet.
**Output:** `update9_results_A.zip` and `update9_results_B.zip`, with the raw answers, scores, logs and `constrained_A.md` / `constrained_B.md`.

## 1. Settings
Edit this cell only.

In [1]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run

# Two runs, one per part: "A" and "B". The parts can run at the same time, in two notebooks.
PART = "A"
MODELS_OF_PART = {"A": ["qwen3-1.7b", "llama-3.2-1b", "llama-3.2-3b"], "B": ["qwen3-4b", "qwen3-8b"]}
# Grouped by model, so each model is downloaded once
CONFIGS = [f"configs/{m}_{p}.yaml" for m in MODELS_OF_PART[PART] for p in ["fp16", "int8", "nf4", "gptq", "awq"]]
CHECK_CONFIGS = [f"configs/{MODELS_OF_PART[PART][0]}_awq.yaml"]
CHECK_LIMIT = 2               # questions per category in the quick check
MODE = "constrained"
REPORT_NAME = f"constrained_{PART}.md"
REPORT_TITLE = f"Constrained decoding, part {PART} (Update 9)"
RUN_TIMEOUT_MIN = 240         # a run that takes longer than this is stopped, and the next one starts
QUANTIZE_TIMEOUT_MIN = 120    # the same for making a checkpoint
DELETE_MODEL_AFTER_RUN = True # frees disk space: deletes each checkpoint after its run, each model after its last run

HF_CHECKPOINT_USER = "shawalkabir"  # who holds the AWQ checkpoints; empty = make them here (about 3 hours more)
# Upload of the GPTQ checkpoints to Hugging Face (needs the HF_WRITE_TOKEN secret)
HF_UPLOAD_USER = "shawalkabir"      # your Hugging Face user name; empty = no upload
HF_UPLOAD_PRIVATE = True

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
BITSANDBYTES_VERSION = "0.45.5"  # same vLLM environment as notebook 04
ACCELERATE_VERSION = "1.6.0"
GPTQMODEL_VERSION = "5.7.0"      # the last GPTQModel release that works with transformers 4.x
QUANT_TORCH_VERSION = "2.9.1"
QUANT_TRANSFORMERS_VERSION = "4.57.6"
QUANT_TORCHAO_VERSION = "0.14.1"  # matches torch 2.9
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # vLLM + BFCL environment (not kept)
QUANT_VENV = os.path.expanduser("~/quant-venv")  # GPTQModel environment (not kept)
CHECKPOINT_DIR = os.path.expanduser("~/checkpoints")   # GPTQ and AWQ checkpoints (not kept: they are large)
os.environ["CHECKPOINT_DIR"] = CHECKPOINT_DIR           # read by src/quantize.py and src/run_eval.py


def run_id_of(config):
    return os.path.splitext(os.path.basename(config))[0] + f"_{MODE}"

## 2. Check the GPU, internet and Hugging Face tokens

In [2]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: NOT SET. Llama 3.2 cannot be downloaded without it (Add-ons -> Secrets).")

if HF_UPLOAD_USER:
    try:
        os.environ["HF_WRITE_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
        print(f"HF_WRITE_TOKEN: loaded; checkpoints will be uploaded to {HF_UPLOAD_USER}/... "
              f"({'private' if HF_UPLOAD_PRIVATE else 'public'})")
    except Exception as e:
        raise RuntimeError("HF_UPLOAD_USER is set but the HF_WRITE_TOKEN secret is missing (Add-ons -> Secrets).") from e
else:
    print("Upload to Hugging Face: off (HF_UPLOAD_USER is empty)")

GPUs:
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

Internet: OK
HF_TOKEN: loaded from Kaggle Secrets
HF_WRITE_TOKEN: loaded; checkpoints will be uploaded to shawalkabir/... (private)


## 3. Get the code and check access to every model
Clones the repository, then downloads one small file of each model at the pinned revision. This finds a missing token or missing Llama approval now, not after an hour of running.

In [3]:
import yaml
from huggingface_hub import hf_hub_download

shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

cfgs = {}
problems = []
for config in CONFIGS:
    with open(f"{REPO}/{config}") as f:
        cfgs[config] = yaml.safe_load(f)
    model, revision = cfgs[config]["model"], str(cfgs[config]["revision"])
    try:
        hf_hub_download(model, "config.json", revision=revision, token=os.environ.get("HF_TOKEN"))
        print(f"  access OK  {model}")
    except Exception as e:
        problems.append(f"{model}: {type(e).__name__}: {str(e)[:200]}")
if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Some models cannot be downloaded. For Llama: add HF_TOKEN under Add-ons -> Secrets, switch it on for "
        "this notebook, and check that https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct says you have access."
    )

Code version: d7abb52 Qwen3-8B AWQ: smaller working chunks (1 GiB chunks ran out of GPU memory at layer 22) (2026-11-03 00:05:34 +0600)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct


## 4. Install the vLLM environment (about 1-5 minutes)
Exactly the same environment as notebook 04: a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]` (vLLM 0.8.5) and the same pins.

In [4]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile "bitsandbytes=={BITSANDBYTES_VERSION}" "accelerate=={ACCELERATE_VERSION}"

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch, bitsandbytes, accelerate; "
    "from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch'), "
    "'| bitsandbytes', version('bitsandbytes'), '| accelerate', version('accelerate')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

Using CPython 3.11.18
Creating virtual environment at: /root/bfcl-venv
Activate with: source /root/bfcl-venv/bin/activate
Using Python 3.11.18 environment at: /root/bfcl-venv
Resolved 215 packages in 2.30s
Prepared 215 packages in 5m 00s
Installed 215 packages in 517ms
 + accelerate==1.6.0
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==2.4.0
 + anyio==4.16.0
 + argcomplete==3.7.2
 + astor==0.8.1
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.3.23
 + bitsandbytes==0.45.5
 + black==26.10.1
 + blake3==1.0.11
 + boto3==1.43.123
 + botocore==1.43.123
 + cachetools==7.2.8
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cohere==5.18.0
 + compressed-tensors==0.9.3
 + cryptography==50.0.2
 + cupy-cuda12x==13.6.0
 + dashscope==2.0.3
 + datamodel-code-generator==0.25.7
 + deprecated==1.3.1
 

## 5. Install the quantization environment (about 2-5 minutes)
A second Python 3.11 environment with GPTQModel. `BUILD_CUDA_EXT=0` skips compiling GPTQModel's own CUDA kernels: they are only needed to run models in GPTQModel, and here vLLM runs them. GPTQModel saves both formats with its plain PyTorch code.

In [5]:
shutil.rmtree(QUANT_VENV, ignore_errors=True)
!uv venv {QUANT_VENV} --python 3.11
!BUILD_CUDA_EXT=0 uv pip install --python {QUANT_VENV}/bin/python "gptqmodel=={GPTQMODEL_VERSION}" "torch=={QUANT_TORCH_VERSION}" "transformers=={QUANT_TRANSFORMERS_VERSION}" "torchao=={QUANT_TORCHAO_VERSION}" pyyaml

QUANT_CHECK = (
    "import gptqmodel, torch, transformers, yaml; from importlib.metadata import version; "
    "print('gptqmodel', version('gptqmodel'), '| torch', version('torch'), '| transformers', version('transformers')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "
    "'| compute capability', torch.cuda.get_device_capability(0))"
)
r = subprocess.run([f"{QUANT_VENV}/bin/python", "-c", QUANT_CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Quantization environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{QUANT_VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze_quant.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze_quant.txt")

Using CPython 3.11.18
Creating virtual environment at: /root/quant-venv
Activate with: source /root/quant-venv/bin/activate
Using Python 3.11.18 environment at: /root/quant-venv
Resolved 74 packages in 413ms
Prepared 42 packages in 4m 55s
Installed 74 packages in 285ms
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + anyio==4.16.0
 + attrs==26.1.0
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + datasets==5.0.1
 + device-smi==0.5.7
 + dill==0.4.1
 + filelock==4.2.7
 + frozenlist==1.8.0
 + fsspec==2026.6.0
 + gptqmodel==5.7.0
 + h11==0.16.0
 + hf-transfer==0.1.9
 + hf-xet==1.8.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==0.36.2
 + idna==3.20
 + jinja2==3.1.6
 + kernels==0.12.3
 + logbar==0.4.15
 + markupsafe==3.0.4
 + maturin==1.16.0
 + mpmath==1.3.0
 + multidict==7.2.1
 + multiprocess==0.70.19
 + networkx==3.6.1
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.

## 6. Quick check: does constrained decoding run on an AWQ checkpoint?
The first model's AWQ checkpoint is downloaded and answers 2 questions per category. The answers go to `check/`, apart from the real results. If this fails, the AWQ runs are skipped in step 7 and the others still run.

In [6]:
def tail(path, n=25):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


def run_logged(args, log_path, timeout_min, venv=VENV):
    """Runs a script of the repository in an environment; returns its exit code, or None after a timeout."""
    with open(log_path, "w") as log:
        try:
            return subprocess.run([f"{venv}/bin/python", *args], cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
                                  timeout=timeout_min * 60).returncode
        except subprocess.TimeoutExpired:
            print(f"  stopped after {timeout_min} minutes")
            return None


def gpu_memory_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def wait_for_free_gpus(limit_mib=1000, wait_s=120):
    """A finished run should leave the GPUs empty; wait briefly if they are not yet."""
    start = time.time()
    while max(gpu_memory_used_mib()) > limit_mib and time.time() - start < wait_s:
        time.sleep(5)
    used = gpu_memory_used_mib()
    if max(used) > limit_mib:
        print(f"  WARNING: GPU memory still in use before this run: {used} MiB")


def make_checkpoint(config, stem, log_path):
    """Downloads an AWQ checkpoint, or makes the INT8 or GPTQ one. Returns True if it worked."""
    method = cfgs[config]["quantize"]["method"]
    if method == "awq" and HF_CHECKPOINT_USER:
        from huggingface_hub import snapshot_download
        repo_id = f"{HF_CHECKPOINT_USER}/{cfgs[config]['model'].split('/')[-1]}-AWQ-W4A16-g128"
        with open(log_path, "w") as log:
            try:
                if not os.path.exists(f"{CHECKPOINT_DIR}/{stem}/quantize_info.json"):
                    snapshot_download(repo_id, local_dir=f"{CHECKPOINT_DIR}/{stem}", token=os.environ.get("HF_TOKEN"))
                log.write(f"Using {repo_id}\n")
                return True
            except Exception as e:
                log.write(f"Could not download {repo_id}: {type(e).__name__}: {str(e)[:300]}\n")
                return False
    venv = VENV if method == "bnb-int8" else QUANT_VENV   # 8-bit bitsandbytes checkpoints come from the vLLM environment
    return run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN, venv) == 0


CHECK_DIR = f"{WORK}/check"
shutil.rmtree(CHECK_DIR, ignore_errors=True)
works = {}
for config in CHECK_CONFIGS:
    cfg, stem = cfgs[config], os.path.splitext(os.path.basename(config))[0]
    print(f"\n=== check {stem}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    ok = True
    if "quantize" in cfg:
        log_path = f"{WORK}/check_quantize_{stem}.log"
        ok = make_checkpoint(config, stem, log_path)
        print("  checkpoint: " + tail(log_path, 1).strip())
        wait_for_free_gpus()
    if ok:
        log_path = f"{WORK}/check_{stem}.log"
        ok = run_logged(["src/run_eval.py", "--config", config, "--mode", MODE, "--limit", str(CHECK_LIMIT),
                         "--results-dir", CHECK_DIR, "--overwrite"], log_path, 30) == 0
    works[cfg["precision"]] = ok
    if not ok:
        print(f"  FAILED. End of {log_path}:\n{tail(log_path, 40)}")
        continue

    run_id = f"{run_id_of(config)}_limit{CHECK_LIMIT}"
    r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"{CHECK_DIR}/raw/{run_id}.jsonl"],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-1500:])
    with open(f"{CHECK_DIR}/meta/{run_id}.json") as f:
        meta = json.load(f)
    print(f"  weights {meta['memory']['model_weights_gib']} GiB, {meta['output_tokens_per_second']} output tokens/s, "
          f"finish reasons {meta['finish_reasons']}")
    with open(f"{CHECK_DIR}/raw/{run_id}.jsonl") as f:
        for line in list(f)[:3]:
            row = json.loads(line)
            print(f"  {row['id']}: {row['output'][:200]!r}")

print("\nQuick check: " + ", ".join(f"{p.upper()} {'works' if ok else 'FAILED'}" for p, ok in works.items()))
if not all(works.values()):
    print("The runs of a failed method are skipped in step 7; the other method still runs.")


=== check qwen3-1.7b_awq: started 20:23:06 UTC


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

  checkpoint: Using shawalkabir/Qwen3-1.7B-AWQ-W4A16-g128

qwen3-1.7b_awq_constrained_limit2  (14 questions)
  category               correct  accuracy   parsed
  simple_python              2/2    100.00   100.00
  simple_java                1/2     50.00   100.00
  simple_javascript          2/2    100.00   100.00
  multiple                   2/2    100.00   100.00
  parallel                   2/2    100.00   100.00
  parallel_multiple          2/2    100.00   100.00
  irrelevance                2/2    100.00   100.00
  saved /kaggle/working/check/scores/qwen3-1.7b_awq_constrained_limit2.jsonl and /kaggle/working/check/summary.csv

  weights 1.2721 GiB, 38.7 output tokens/s, finish reasons {'stop': 14}
  simple_python_0: '[{"name": "calculate_triangle_area", "arguments": {"base": 10, "height": 5}}]'
  simple_python_1: '[{"name": "math.factorial", "arguments": {"number": 5}}]'
  simple_java_0: '[{"name": "GeometryPresentation.createPresentation", "arguments": {"controller": "mapControl

## 7. Run and score every config (several hours)
One config at a time: the checkpoint (`make_checkpoint`, not needed for FP16 and NF4), `run_eval.py` (answers), then `score.py` (accuracy). Every step's full log is saved. If a config fails, the notebook prints the end of its log and goes on with the next one. Each GPTQ checkpoint is uploaded after its run.

In [7]:
def upload_checkpoint(config, stem):
    """Uploads a finished checkpoint to Hugging Face; returns (repo id, commit hash)."""
    from huggingface_hub import HfApi
    name = cfgs[config]["model"].split("/")[-1]
    repo_id = f"{HF_UPLOAD_USER}/{name}-{cfgs[config]['precision'].upper()}-W4A16-g128"
    api = HfApi(token=os.environ["HF_WRITE_TOKEN"])
    api.create_repo(repo_id, private=HF_UPLOAD_PRIVATE, exist_ok=True)
    commit = api.upload_folder(folder_path=f"{CHECKPOINT_DIR}/{stem}", repo_id=repo_id,
                               commit_message=f"Made by src/quantize.py --config {config} ({REPO_URL})")
    return repo_id, commit.oid


status, minutes, uploads = {}, {}, {}
last_config_of = {cfgs[c]["model"]: c for c in CONFIGS}   # each model is deleted after its last config
for config in CONFIGS:
    cfg, run_id = cfgs[config], run_id_of(config)
    stem = os.path.splitext(os.path.basename(config))[0]
    if not works.get(cfg["precision"], True):
        status[run_id] = "SKIPPED (quick check failed)"
    else:
        print(f"\n=== {run_id} ({cfg['model']}): started {time.strftime('%H:%M:%S')} UTC", flush=True)
        start = time.time()
        ok = True
        if "quantize" in cfg:
            wait_for_free_gpus()
            log_path = f"{WORK}/quantize_{stem}.log"
            ok = make_checkpoint(config, stem, log_path)
            if ok:
                print("  checkpoint: " + tail(log_path, 1).strip())
            else:
                status[run_id] = "QUANTIZE FAILED"
                print(f"  quantization FAILED. End of {log_path}:\n{tail(log_path)}")
        if ok:
            wait_for_free_gpus()
            log_path = f"{WORK}/run_{run_id}.log"
            run_logged(["src/run_eval.py", "--config", config, "--mode", MODE, "--overwrite"], log_path,
                       RUN_TIMEOUT_MIN)
            if not os.path.exists(f"{REPO}/results/meta/{run_id}.json"):
                status[run_id] = "FAILED"
                print(f"  FAILED. End of {log_path}:\n{tail(log_path)}")
            else:
                print("  answers: " + next((l.strip() for l in tail(log_path, 10).splitlines()
                                            if l.startswith("Done:")), ""))
                r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"results/raw/{run_id}.jsonl"],
                                   cwd=REPO, capture_output=True, text=True)
                print(r.stdout[-2000:])
                status[run_id] = "OK" if r.returncode == 0 else "SCORING FAILED"
                if r.returncode != 0:
                    print(r.stderr[-3000:])
        minutes[run_id] = (time.time() - start) / 60
        print(f"  {status[run_id]} after {minutes[run_id]:.1f} min")
        if HF_UPLOAD_USER and cfg["precision"] == "gptq" and os.path.exists(f"{CHECKPOINT_DIR}/{stem}/quantize_info.json"):
            try:
                uploads[stem] = upload_checkpoint(config, stem)
                print(f"  uploaded to https://huggingface.co/{uploads[stem][0]} (commit {uploads[stem][1][:7]})")
            except Exception as e:
                print(f"  upload FAILED: {type(e).__name__}: {str(e)[:300]}")

    if DELETE_MODEL_AFTER_RUN:
        shutil.rmtree(f"{CHECKPOINT_DIR}/{stem}", ignore_errors=True)
        if last_config_of[cfg["model"]] == config:
            shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{cfg['model'].replace('/', '--')}"),
                          ignore_errors=True)

print("\nSummary:")
for run_id, s in status.items():
    took = f"  ({minutes[run_id]:.1f} min)" if run_id in minutes else ""
    print(f"  {run_id:28s} {s}{took}")
print(f"  total: {sum(minutes.values()) / 60:.1f} hours")
if any(s != "OK" for s in status.values()):
    print("Some runs did not finish. The results of the others are still reported and packaged below.")
if uploads:
    with open(f"{WORK}/hf_uploads.json", "w") as f:
        json.dump({stem: {"repo": repo, "commit": commit} for stem, (repo, commit) in uploads.items()}, f, indent=2)


=== qwen3-1.7b_fp16_constrained (Qwen/Qwen3-1.7B): started 20:24:41 UTC
  answers: Done: 1390 answers in 974 s (88.9 output tokens/s); finish reasons {'stop': 1386, 'length': 4}

qwen3-1.7b_fp16_constrained  (1390 questions)
  category               correct  accuracy   parsed
  simple_python          374/400     93.50    99.75
  simple_java             65/100     65.00    99.00
  simple_javascript        33/50     66.00    98.00
  multiple               182/200     91.00    99.50
  parallel               168/200     84.00   100.00
  parallel_multiple      160/200     80.00   100.00
  irrelevance            186/240     77.50   100.00
  simple_ast                         74.83         
  non_live_overall                   82.46         
  saved /kaggle/working/repo/results/scores/qwen3-1.7b_fp16_constrained.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 17.7 min

=== qwen3-1.7b_int8_constrained (Qwen/Qwen3-1.7B): started 20:42:23 UTC
  checkpoint: Done in 16 s: 1.909 GiB

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

  checkpoint: Using shawalkabir/Llama-3.2-1B-Instruct-AWQ-W4A16-g128
  answers: Done: 1390 answers in 949 s (175.8 output tokens/s); finish reasons {'stop': 1364, 'length': 26}

llama-3.2-1b_awq_constrained  (1390 questions)
  category               correct  accuracy   parsed
  simple_python          272/400     68.00    99.50
  simple_java             51/100     51.00    99.00
  simple_javascript        32/50     64.00    98.00
  multiple                98/200     49.00   100.00
  parallel                95/200     47.50    98.00
  parallel_multiple       67/200     33.50    99.00
  irrelevance            122/240     50.83    92.92
  simple_ast                         61.00         
  non_live_overall                   47.75         
  saved /kaggle/working/repo/results/scores/llama-3.2-1b_awq_constrained.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 17.1 min

=== llama-3.2-3b_fp16_constrained (meta-llama/Llama-3.2-3B-Instruct): started 00:04:43 UTC
  answers: Done: 1

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

  checkpoint: Using shawalkabir/Llama-3.2-3B-Instruct-AWQ-W4A16-g128
  answers: Done: 1390 answers in 935 s (73.3 output tokens/s); finish reasons {'stop': 1389, 'length': 1}

llama-3.2-3b_awq_constrained  (1390 questions)
  category               correct  accuracy   parsed
  simple_python          355/400     88.75   100.00
  simple_java             58/100     58.00   100.00
  simple_javascript        24/50     48.00   100.00
  multiple               169/200     84.50   100.00
  parallel               129/200     64.50   100.00
  parallel_multiple      124/200     62.00    99.50
  irrelevance            167/240     69.58   100.00
  simple_ast                         64.92         
  non_live_overall                   68.98         
  saved /kaggle/working/repo/results/scores/llama-3.2-3b_awq_constrained.jsonl and /kaggle/working/repo/results/summary.csv

  OK after 17.2 min

Summary:
  qwen3-1.7b_fp16_constrained  OK  (17.7 min)
  qwen3-1.7b_int8_constrained  OK  (32.4 min)
  qwen3-1.

## 8. Tables for all five precisions
**Overall** is BFCL's Non-Live Overall Acc (mean of simple, multiple, parallel and parallel-multiple); irrelevance is reported separately. **Parsed** should be 100% or close to it here, because the schema forces the format. The comparison with free decoding comes in Update 10, from `results/summary.csv`.

In [8]:
import re

import pandas as pd

COLUMNS = {
    "simple_python": "Python", "simple_java": "Java", "simple_javascript": "JavaScript",
    "multiple": "Multiple", "parallel": "Parallel", "parallel_multiple": "Parallel Multiple",
    "non_live_overall": "Overall", "irrelevance": "Irrelevance",
}
PRECISIONS = ["fp16", "int8", "nf4", "gptq", "awq"]


def table_order(base):
    family = 0 if "qwen" in base else 1
    size = float(re.search(r"(\d+(?:\.\d+)?)b", base).group(1))
    return family, size


bases = sorted({os.path.basename(c).rsplit("_", 1)[0] for c in CONFIGS}, key=table_order)  # e.g. qwen3-1.7b
summary = pd.read_csv(f"{REPO}/results/summary.csv")
acc_rows, info_rows, overall, metas = [], [], {}, {}
for base in bases:
    for precision in PRECISIONS:
        run_id = f"{base}_{precision}_{MODE}"
        s = summary[summary["run_id"] == run_id].set_index("category")
        if s.empty or "non_live_overall" not in s.index:
            continue
        with open(f"{REPO}/results/meta/{run_id}.json") as f:
            meta = metas[run_id] = json.load(f)
        scores = pd.read_json(f"{REPO}/results/scores/{run_id}.jsonl", lines=True)
        ast = scores[scores["category"] != "irrelevance"]
        name, label = meta["model"].split("/")[-1], precision.upper()
        overall.setdefault(name, {})[label] = float(s.loc["non_live_overall", "accuracy"])
        acc_rows.append({"Model": name, "Precision": label,
                         **{col: float(s.loc[c, "accuracy"]) for c, col in COLUMNS.items()}})
        info_rows.append({
            "Model": name,
            "Precision": label,
            "Parsed, AST questions (%)": round(100 * ast["parse_ok"].mean(), 2),
            "Unparsed answers": int((~ast["parse_ok"]).sum()),
            "Cut off at max_tokens": meta["finish_reasons"].get("length", 0),
            "GPUs": meta["config"]["tensor_parallel_size"],
            "Weights per GPU (GiB)": meta["memory"]["model_weights_gib"],
            "Generation (s)": meta["generation_seconds"],
            "Output tokens/s": meta["output_tokens_per_second"],
        })

compare_rows = []
for name, acc in overall.items():
    row = {"Model": name, **{p.upper(): acc.get(p.upper()) for p in PRECISIONS}}
    for p in ["INT8", "NF4", "GPTQ", "AWQ"]:
        row[f"{p} - FP16"] = round(acc[p] - acc["FP16"], 2) if p in acc and "FP16" in acc else None
    compare_rows.append(row)

compare = pd.DataFrame(compare_rows)
accuracy = pd.DataFrame(acc_rows)
info = pd.DataFrame(info_rows)
display(compare)
display(accuracy)
display(info)
compare.to_csv(f"{WORK}/constrained_overall_{PART}.csv", index=False)
accuracy.to_csv(f"{WORK}/constrained_accuracy_{PART}.csv", index=False)
info.to_csv(f"{WORK}/constrained_run_info_{PART}.csv", index=False)

if not info.empty:
    quantized = info[info["Precision"] != "FP16"]
    missing = quantized[quantized["Weights per GPU (GiB)"].isna() | quantized["Output tokens/s"].isna()]
    print("Memory and speed recorded for every quantized run." if missing.empty else
          f"WARNING: memory or speed missing for:\n{missing[['Model', 'Precision']]}")

r = subprocess.run([f"{VENV}/bin/python", "src/main_table.py", "--mode", MODE], cwd=REPO, capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr[-3000:])

,Model,FP16,INT8,NF4,GPTQ,AWQ,INT8 - FP16,NF4 - FP16,GPTQ - FP16,AWQ - FP16
0,Qwen3-1.7B,82.46,82.79,78.50,71.65,80.81,0.33,-3.96,-10.81,-1.65
1,Llama-3.2-1B-Instruct,48.19,45.50,17.83,2.90,47.75,-2.69,-30.36,-45.29,-0.44
2,Llama-3.2-3B-Instruct,71.04,69.42,65.96,64.71,68.98,-1.62,-5.08,-6.33,-2.06


,Model,Precision,Python,Java,JavaScript,Multiple,Parallel,Parallel Multiple,Overall,Irrelevance
0,Qwen3-1.7B,FP16,93.50,65.0,66.0,91.0,84.0,80.0,82.46,77.50
1,Qwen3-1.7B,INT8,92.50,65.0,68.0,90.0,84.5,81.5,82.79,72.50
2,Qwen3-1.7B,NF4,89.00,67.0,66.0,90.0,78.5,71.5,78.50,80.83
3,Qwen3-1.7B,GPTQ,83.25,61.0,72.0,63.5,76.5,74.5,71.65,56.25
4,Qwen3-1.7B,AWQ,92.25,65.0,70.0,87.5,81.5,78.5,80.81,69.58
5,Llama-3.2-1B-Instruct,FP16,50.75,33.0,52.0,53.5,55.0,39.0,48.19,84.17
6,Llama-3.2-1B-Instruct,INT8,45.00,27.0,36.0,55.0,52.0,39.0,45.50,80.83
7,Llama-3.2-1B-Instruct,NF4,20.50,1.0,2.0,53.5,5.0,5.0,17.83,77.08
8,Llama-3.2-1B-Instruct,GPTQ,0.25,0.0,0.0,9.0,0.5,2.0,2.90,99.58
9,Llama-3.2-1B-Instruct,AWQ,68.00,51.0,64.0,49.0,47.5,33.5,47.75,50.83


,Model,Precision,"Parsed, AST questions (%)",Unparsed answers,Cut off at max_tokens,GPUs,Weights per GPU (GiB),Generation (s),Output tokens/s
0,Qwen3-1.7B,FP16,99.65,4,4,1,3.2152,974.1,88.9
1,Qwen3-1.7B,INT8,99.83,2,2,1,1.9027,1876.9,41.9
2,Qwen3-1.7B,NF4,99.91,1,2,1,1.2674,1964.8,39.1
3,Qwen3-1.7B,GPTQ,99.91,1,1,1,1.2721,919.6,86.7
4,Qwen3-1.7B,AWQ,99.91,1,1,1,1.2721,946.5,79.8
5,Llama-3.2-1B-Instruct,FP16,99.30,8,16,1,2.3185,834.2,143.3
6,Llama-3.2-1B-Instruct,INT8,99.04,11,16,1,1.4123,1165.6,100.2
7,Llama-3.2-1B-Instruct,NF4,99.83,2,8,1,0.9785,1381.0,39.8
8,Llama-3.2-1B-Instruct,GPTQ,100.00,0,0,1,0.9817,712.1,8.7
9,Llama-3.2-1B-Instruct,AWQ,99.13,10,26,1,0.9817,949.0,175.8


Memory and speed recorded for every quantized run.
# Main Table: Accuracy by Precision (constrained decoding)

BFCL `non_live` (1,390 questions per run), our prompt format and scorer (`src/run_eval.py`, `src/score.py`), greedy decoding. Built from `results/summary.csv` by `src/main_table.py`; a missing run is shown as –.

## Non-Live Overall accuracy (%)

BFCL's Non-Live Overall Acc: the unweighted mean of simple (itself the mean of Python, Java and JavaScript), multiple, parallel and parallel-multiple.

| Model | FP16 | INT8 | NF4 | GPTQ | AWQ |
|---|---:|---:|---:|---:|---:|
| Qwen3-1.7B | 82.46 | 82.79 | 78.50 | 71.65 | 80.81 |
| Qwen3-4B | – | – | – | – | – |
| Qwen3-8B | – | – | – | – | – |
| Llama-3.2-1B-Instruct | 48.19 | 45.50 | 17.83 | 2.90 | 47.75 |
| Llama-3.2-3B-Instruct | 71.04 | 69.42 | 65.96 | 64.71 | 68.98 |

## Change from FP16 (percentage points)

| Model | INT8 | NF4 | GPTQ | AWQ |
|---|---:|---:|---:|---:|
| Qwen3-1.7B | +0.33 | -3.96 | -10.81 | -1.65 |
| Qwen3-4B |

## 9. Write the report and package the results

In [9]:
import glob


def md_table(df):
    def fmt(v):
        if v is None or (isinstance(v, float) and v != v):
            return ""
        return f"{v:.2f}" if isinstance(v, float) else str(v)
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" if i == 0 else "---:" for i in range(len(df.columns))) + "|"
    return [head, sep] + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]


first = next((m for m in metas.values() if (m.get("quantized_checkpoint") or {}).get("calibration")), None)
lines = [f"# {REPORT_TITLE}", "", f"**Date:** {date.today()}",
         "**Goal:** function-calling accuracy of every core model at every precision when the answer is forced to "
         "match the JSON schema (vLLM guided decoding, xgrammar).", ""]
if first:
    v = first["versions"]
    lines += [
        "## Setup", "",
        f"- **Questions:** BFCL `non_live`, {first['n_questions']} questions per run, bfcl-eval {v['bfcl-eval']}",
        f"- **Prompt format:** `{first['prompt_format']}` (as in Updates 3-4), {MODE} decoding; Qwen3 with thinking off",
        f"- **Decoding:** greedy (temperature 0), seed {first['config']['sampling']['seed']}, "
        f"at most {first['config']['sampling']['max_tokens']} answer tokens",
    ]
    q = first["quantized_checkpoint"]
    c, qv = q["calibration"], q["versions"]
    gptq = next((m["quantized_checkpoint"]["settings"] for m in metas.values() if m["precision"] == "gptq"), None)
    awq = next((m["quantized_checkpoint"]["settings"] for m in metas.values() if m["precision"] == "awq"), None)
    lines += [
        f"- **Inference:** vLLM {v['vllm']} on one {first['gpus'][0]} per run (FP16 Qwen3-8B used two), "
        "with its GPTQ and AWQ GEMM kernels (the faster Marlin kernels need a newer GPU)",
        f"- **Quantization:** GPTQModel {qv['gptqmodel']} (torch {qv['torch']}, transformers {qv['transformers']}), "
        "`src/quantize.py`; the same pinned model revisions as FP16",
        f"- **Calibration (same for every model and both methods):** the first {c['samples']} documents with at least "
        f"{c['min_chars']:,} characters from `{c['dataset']}` `{c['file']}` (revision `{c['revision'][:7]}`), "
        f"each cut to {c['tokens']} tokens",
    ]
    if gptq:
        lines.append(f"- **GPTQ:** {gptq['bits']}-bit weights, groups of {gptq['group_size']}, "
                     f"{'symmetric' if gptq['sym'] else 'asymmetric'}, desc_act={gptq['desc_act']}; "
                     "saved in the GPTQ format")
    if awq:
        lines.append(f"- **AWQ:** {awq['bits']}-bit weights, groups of {awq['group_size']}, "
                     f"{'with' if awq['zero_point'] else 'without'} zero points; saved in the AWQ GEMM format")
    lines += [
        f"- **Software:** Python {v['python']}, torch {v['torch']}, transformers {v['transformers']}",
        f"- **Code:** commit `{(first['code']['commit'] or 'unknown')[:7]}`; model revisions are in `results/meta/`",
        "- **Overall** = BFCL's Non-Live Overall Acc (unweighted mean of simple, multiple, parallel and "
        "parallel-multiple; simple = mean of Python, Java, JavaScript). Irrelevance is reported on its own.",
        "",
        "## Overall accuracy (%)", "",
    ] + md_table(compare) + [
        "", "## Accuracy per category (%)", "",
    ] + md_table(accuracy) + ["", "## Format, length and speed", ""] + md_table(info)
failed = [r for r, s in status.items() if s != "OK"]
lines += ["", "## Notes", ""]
lines += [f"- Did not finish: {', '.join(failed)}" if failed else "- All runs finished."]
if uploads:
    lines += [f"- Checkpoints on Hugging Face: " + ", ".join(f"[{repo}](https://huggingface.co/{repo}) (`{commit[:7]}`)"
                                                         for repo, commit in uploads.values())]
lines += ["- "]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/{REPORT_NAME}", "w") as f:
    f.write(report)
print(report)

pack = f"{WORK}/update9_results_{PART}"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results", ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
if os.path.exists(CHECK_DIR):
    shutil.copytree(CHECK_DIR, f"{pack}/check")
for name in [REPORT_NAME, f"constrained_overall_{PART}.csv", f"constrained_accuracy_{PART}.csv", f"constrained_run_info_{PART}.csv",
             "requirements-freeze.txt", "requirements-freeze_quant.txt", "hf_uploads.json"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
for log_path in sorted(glob.glob(f"{WORK}/*.log")):
    shutil.copy(log_path, pack)
archive = shutil.make_archive(pack, "zip", pack)
print("Saved", archive)

# Constrained decoding, part A (Update 9)

**Date:** 2026-11-03
**Goal:** function-calling accuracy of every core model at every precision when the answer is forced to match the JSON schema (vLLM guided decoding, xgrammar).

## Setup

- **Questions:** BFCL `non_live`, 1390 questions per run, bfcl-eval 2026.3.23
- **Prompt format:** `json-array-v1` (as in Updates 3-4), constrained decoding; Qwen3 with thinking off
- **Decoding:** greedy (temperature 0), seed 42, at most 4096 answer tokens
- **Inference:** vLLM 0.8.5 on one Tesla T4 per run (FP16 Qwen3-8B used two), with its GPTQ and AWQ GEMM kernels (the faster Marlin kernels need a newer GPU)
- **Quantization:** GPTQModel 5.7.0 (torch 2.9.1, transformers 4.57.6), `src/quantize.py`; the same pinned model revisions as FP16
- **Calibration (same for every model and both methods):** the first 256 documents with at least 3,000 characters from `allenai/c4` `en/c4-train.00001-of-01024.json.gz` (revision `1588ec4`), each cut to 512 tokens
- **

## After the run: what goes into the repository

1. Download **`update9_results_A.zip`** and **`update9_results_B.zip`** from the Output tab of each saved version, and each notebook with its outputs.
2. Merge the two parts: `results/meta/`, `results/scores/` and the new rows of `results/summary.csv`; the raw answers in `results/raw/` stay out of git.
3. Run `python src/main_table.py --mode constrained` and `python src/taxonomy.py --mode constrained`.
4. Add the GPU time to `notes/compute_log.md`, any design choice to `notes/decisions.md`, and the entry to `UPDATES.md`, then commit and push.